# Figure 8 — inferred $Q_f$ and $D_{it}$ during annealing

This notebook reproduces **Figure 8** directly from the archived Monte Carlo fit results in `data/inp_annealing_trajectory_batched_wide_asdep.csv`.

Each point is one fit. Lower-RMS fits are plotted larger and more opaquely. Black markers and error bars show the median and the 16th–84th percentile interval. The as-deposited fits are retained as gray diagnostic points even though they do not meet the compatibility threshold; compatible annealed fits are colored by temperature.

To generate additional fits, use `monte_carlo_annealing_inference.ipynb`. Do not silently mix results obtained with a different scientific configuration ID.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

HERE = Path.cwd()
if not (HERE / 'data').exists():
    candidate = HERE / 'notebooks' / 'paper_inp_pox_alox'
    if (candidate / 'data').exists():
        HERE = candidate

DATA_PATH = HERE / 'data' / 'inp_annealing_trajectory_batched_wide_asdep.csv'
OUTPUT_DIR = HERE / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

mc = pd.read_csv(DATA_PATH)
mc['compatible'] = mc['compatible'].astype(str).str.lower().eq('true')
print(f'{len(mc)} state fits from {mc.draw_id.nunique()} Monte Carlo draws')
print('Configuration IDs:', ', '.join(map(str, mc.config_id.dropna().unique())))


## Select fits and define the graphical encoding

Annealed points must pass the stored compatibility criterion. The as-deposited state is shown separately as a model-mismatch diagnostic because its PL pair is not reproduced within the adopted two-parameter surface model.


In [ ]:
STATE_ORDER = [
    'As-deposited', '150 C anneal', '200 C anneal', '250 C anneal',
    '300 C anneal', '350 C anneal',
]
COLORS = {
    'As-deposited': '0.35',
    '150 C anneal': 'tab:purple',
    '200 C anneal': 'tab:blue',
    '250 C anneal': 'tab:green',
    '300 C anneal': 'tab:orange',
    '350 C anneal': 'tab:red',
}
DISPLAY_LABELS = {
    'As-deposited': 'As-deposited',
    '150 C anneal': r'150 $^\circ$C',
    '200 C anneal': r'200 $^\circ$C',
    '250 C anneal': r'250 $^\circ$C',
    '300 C anneal': r'300 $^\circ$C',
    '350 C anneal': r'350 $^\circ$C',
}

missing = sorted(set(STATE_ORDER) - set(mc.state.unique()))
if missing:
    raise ValueError(f'Missing manuscript states in CSV: {missing}')

selected = pd.concat([
    mc.loc[mc.state.eq('As-deposited')],
    mc.loc[mc.state.ne('As-deposited') & mc.compatible],
], ignore_index=True)

rms = selected.rms_log_residual.to_numpy(float)
scale = np.quantile(rms[np.isfinite(rms)], [0.05, 0.95])
quality = 1 - np.clip((rms - scale[0]) / max(scale[1] - scale[0], 1e-12), 0, 1)
selected['marker_size'] = 12 + 34 * quality
selected['marker_alpha'] = 0.18 + 0.62 * quality

# Deterministic jitter: stable across reruns and independent of row order.
rng = np.random.default_rng(20261002)
selected['x_jitter'] = rng.uniform(-0.16, 0.16, len(selected))
selected.head()


## Publication figure

In [ ]:
plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 600, 'font.size': 10,
    'axes.linewidth': 1.5, 'lines.linewidth': 2.0,
    'xtick.top': True, 'ytick.right': True,
})

fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.3), sharex=True,
                         constrained_layout=True)

for i, state in enumerate(STATE_ORDER):
    subset = selected.loc[selected.state.eq(state)]
    for _, row in subset.iterrows():
        x = i + row.x_jitter
        style = dict(s=row.marker_size, alpha=row.marker_alpha,
                     color=COLORS[state], edgecolor='none', zorder=2)
        axes[0].scatter(x, row.qf_over_q_fit_cm2 / 1e12, **style)
        axes[1].scatter(x, row.dit_fit, **style)

    # Summary of the displayed cloud; percentiles are preferable to mean ± SD
    # because the inverse distributions are non-Gaussian and sometimes skewed.
    for ax, values in [
        (axes[0], subset.qf_over_q_fit_cm2.to_numpy(float) / 1e12),
        (axes[1], subset.dit_fit.to_numpy(float)),
    ]:
        median = np.nanmedian(values)
        p16, p84 = np.nanpercentile(values, [16, 84])
        ax.errorbar(i, median, yerr=[[median-p16], [p84-median]],
                    fmt='o', ms=5.5, mfc='white', mec='black', mew=1.1,
                    ecolor='black', elinewidth=1.3, capsize=3.5, zorder=5)

axes[0].axhline(0, color='0.55', lw=0.9, ls=':')
axes[0].set_ylabel(r'$Q_f/q$ ($10^{12}$ cm$^{-2}$)')
axes[1].set_ylabel(r'$D_{it}$ (eV$^{-1}$ cm$^{-2}$)')
axes[1].set_yscale('log')
axes[1].set_xticks(range(len(STATE_ORDER)),
                   [DISPLAY_LABELS[s] for s in STATE_ORDER],
                   rotation=24, ha='right')

for ax in axes:
    ax.set_xlim(-0.5, len(STATE_ORDER)-0.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.tick_params(width=1.5)

fig.savefig(OUTPUT_DIR / 'figure_08_annealing_inference.png', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_08_annealing_inference.pdf', bbox_inches='tight')
plt.show()


## Reporting note

The black interval is the central 68% empirical interval (16th–84th percentiles), not a Gaussian standard deviation. The gray as-deposited cloud should be described as diagnostic rather than as a quantitatively compatible inference.
